# Telecom Churn Prediction
This project focuses on predicting customer churn for a telecom company using machine learning. The goal is to build a classification model capable of identifying churning customers with a high focus on **Recall** to minimize customer loss, while maintaining an acceptable level of precision.

In [1]:
import pandas as pd
import numpy as np

## 1. Exploratory Data Analysis & Class Balance Check
Let's load the dataset and analyze its structure and class distribution to understand if we are dealing with an imbalanced classification problem.

In [2]:
from sklearn.datasets import fetch_openml

churn_data = fetch_openml(name='churn', version=1, as_frame=True)
df_telecom = churn_data.frame

In [4]:
print(df_telecom.columns.tolist())

['state', 'account_length', 'area_code', 'phone_number', 'international_plan', 'voice_mail_plan', 'number_vmail_messages', 'total_day_minutes', 'total_day_calls', 'total_day_charge', 'total_eve_minutes', 'total_eve_calls', 'total_eve_charge', 'total_night_minutes', 'total_night_calls', 'total_night_charge', 'total_intl_minutes', 'total_intl_calls', 'total_intl_charge', 'number_customer_service_calls', 'class']


In [5]:
print("Class balance (0 = Stayed, 1 = Churned):")
print(df_telecom['class'].value_counts(normalize=True).round(4))

Class balance (0 = Stayed, 1 = Churned):
0    0.8586
1    0.1414
Name: class, dtype: float64


In [6]:
df_telecom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   state                          5000 non-null   float64 
 1   account_length                 5000 non-null   float64 
 2   area_code                      5000 non-null   category
 3   phone_number                   5000 non-null   float64 
 4   international_plan             5000 non-null   category
 5   voice_mail_plan                5000 non-null   category
 6   number_vmail_messages          5000 non-null   float64 
 7   total_day_minutes              5000 non-null   float64 
 8   total_day_calls                5000 non-null   float64 
 9   total_day_charge               5000 non-null   float64 
 10  total_eve_minutes              5000 non-null   float64 
 11  total_eve_calls                5000 non-null   float64 
 12  total_eve_charge               500

## 2. Data Preprocessing & Feature Engineering
We will drop redundant numerical columns (`charge` features) because they are perfectly correlated with `minutes`, along with non-informative identifiers. We will also convert categorical and binary columns into proper integer types.

In [7]:

drop_cols = ['phone_number', 'area_code', 'total_day_charge', 'total_eve_charge', 'total_night_charge', 'total_intl_charge']
df_clean = df_telecom.drop(columns=drop_cols)


df_clean['target'] = df_clean['class'].map({'yes': 1, 'no': 0, 'True': 1, 'False': 0, '1': 1, '0': 0})

if df_clean['target'].isnull().all():
    df_clean['target'] = df_clean['class'].astype(int)

df_clean = df_clean.drop(columns=['class'])

In [8]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   state                          5000 non-null   float64 
 1   account_length                 5000 non-null   float64 
 2   international_plan             5000 non-null   category
 3   voice_mail_plan                5000 non-null   category
 4   number_vmail_messages          5000 non-null   float64 
 5   total_day_minutes              5000 non-null   float64 
 6   total_day_calls                5000 non-null   float64 
 7   total_eve_minutes              5000 non-null   float64 
 8   total_eve_calls                5000 non-null   float64 
 9   total_night_minutes            5000 non-null   float64 
 10  total_night_calls              5000 non-null   float64 
 11  total_intl_minutes             5000 non-null   float64 
 12  total_intl_calls               500

In [9]:
df_clean['international_plan'] = df_clean['international_plan'].astype(int)
df_clean['voice_mail_plan'] = df_clean['voice_mail_plan'].astype(int)
df_clean['target'] = df_clean['target'].astype(int)

df_clean['number_customer_service_calls'] = df_clean['number_customer_service_calls'].astype(int)

print(df_clean.dtypes)

state                            float64
account_length                   float64
international_plan                 int32
voice_mail_plan                    int32
number_vmail_messages            float64
total_day_minutes                float64
total_day_calls                  float64
total_eve_minutes                float64
total_eve_calls                  float64
total_night_minutes              float64
total_night_calls                float64
total_intl_minutes               float64
total_intl_calls                 float64
number_customer_service_calls      int32
target                             int32
dtype: object


## 3. Train/Test Split
We split the data into training and testing sets (80/20). Stratification is applied to preserve the minority class distribution across both sets.

In [10]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['target'])
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (4000, 14), Test: (1000, 14)


## 4. Baseline Model Evaluation
We evaluate multiple models using their default configurations to establish a baseline. We will look at Logistic Regression (with and without class balancing), Random Forest, and HistGradientBoosting.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model_lr = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))

model_lr.fit(X_train, y_train)

y_pred = model_lr.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.98      0.93       859
           1       0.62      0.20      0.30       141

    accuracy                           0.87      1000
   macro avg       0.75      0.59      0.61      1000
weighted avg       0.85      0.87      0.84      1000



In [12]:
model_lr_balanced = make_pipeline(
    StandardScaler(), 
    LogisticRegression(random_state=42, class_weight='balanced')
)

model_lr_balanced.fit(X_train, y_train)
y_pred_balanced = model_lr_balanced.predict(X_test)

print(classification_report(y_test, y_pred_balanced).split('\n\n')[1])

           0       0.94      0.78      0.86       859
           1       0.35      0.71      0.47       141


In [13]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42, class_weight='balanced')
model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf).split('\n\n')[1])

           0       0.94      0.99      0.97       859
           1       0.92      0.63      0.75       141


In [14]:
from sklearn.ensemble import HistGradientBoostingClassifier

model_gb = HistGradientBoostingClassifier(random_state=42)
model_gb.fit(X_train, y_train)

y_pred_gb = model_gb.predict(X_test)

print(classification_report(y_test, y_pred_gb).split('\n\n')[1])

           0       0.96      0.99      0.98       859
           1       0.92      0.77      0.83       141


## 5. Feature Selection
To optimize the model and reduce noise, we evaluate feature importances using Permutation Importance on the test set and select the top 8 most influential features.

In [15]:
from sklearn.inspection import permutation_importance

In [16]:
permutation_importance(model_gb, X_test, y_test)

{'importances_mean': array([ 0.0012,  0.0022,  0.0504,  0.0286,  0.0012,  0.0998,  0.0012,
         0.0358, -0.0008,  0.0104,  0.0002,  0.0198,  0.0248,  0.0448]),
 'importances_std': array([0.0009798 , 0.00074833, 0.00454313, 0.00205913, 0.0009798 ,
        0.00581034, 0.00116619, 0.0036    , 0.00074833, 0.00233238,
        0.00146969, 0.00386782, 0.00391918, 0.00584466]),
 'importances': array([[ 0.001,  0.   ,  0.001,  0.003,  0.001],
        [ 0.002,  0.003,  0.003,  0.002,  0.001],
        [ 0.052,  0.055,  0.045,  0.055,  0.045],
        [ 0.031,  0.025,  0.028,  0.03 ,  0.029],
        [ 0.003,  0.001,  0.   ,  0.001,  0.001],
        [ 0.099,  0.1  ,  0.11 ,  0.092,  0.098],
        [ 0.002,  0.002, -0.001,  0.002,  0.001],
        [ 0.034,  0.042,  0.031,  0.036,  0.036],
        [-0.002,  0.   , -0.001,  0.   , -0.001],
        [ 0.009,  0.009,  0.015,  0.009,  0.01 ],
        [ 0.   ,  0.003, -0.001, -0.001,  0.   ],
        [ 0.017,  0.019,  0.02 ,  0.027,  0.016],
        

In [17]:
feat = pd.Series(permutation_importance(model_gb, X_test, y_test)['importances_mean'], index=df_clean.drop('target', axis=1).columns).sort_values(ascending=False)

In [18]:
feat.round(3)

total_day_minutes                0.094
number_customer_service_calls    0.050
international_plan               0.050
total_eve_minutes                0.036
voice_mail_plan                  0.029
total_intl_calls                 0.026
total_intl_minutes               0.018
total_night_minutes              0.007
account_length                   0.001
number_vmail_messages            0.001
total_day_calls                  0.001
total_night_calls                0.001
state                            0.000
total_eve_calls                 -0.001
dtype: float64

In [19]:
top_features = ['total_day_minutes', 'number_customer_service_calls', 'international_plan', 'total_eve_minutes', 'total_intl_calls', 'voice_mail_plan', 'total_intl_minutes', 'total_night_minutes']
X_train_slim = X_train[top_features]
X_test_slim = X_test[top_features]

In [20]:
model_slim = HistGradientBoostingClassifier(random_state=42)
model_slim.fit(X_train_slim, y_train)

y_pred_slim = model_slim.predict(X_test_slim)

# Смотрим результат
print(classification_report(y_test, y_pred_slim).split('\n\n')[1])

           0       0.97      0.99      0.98       859
           1       0.91      0.79      0.84       141


## 6. Handling Class Imbalance & Threshold Optimization
Since default boosting models tend to underestimate minority class probabilities, we retrain the `HistGradientBoostingClassifier` using calculated sample weights. We then fine-tune the decision threshold to achieve a high **Recall (~88%)** while optimizing **Precision** to prevent excessive false alarms for the business.

In [21]:
from sklearn.utils.class_weight import compute_sample_weight


sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)


model_slim_balanced = HistGradientBoostingClassifier(random_state=42)
model_slim_balanced.fit(X_train_slim, y_train, sample_weight=sample_weights)


probs_balanced = model_slim_balanced.predict_proba(X_test_slim)[:, 1]


for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]:
    y_pred_custom = (probs_balanced > threshold).astype(int)
    
    report = classification_report(y_test, y_pred_custom, output_dict=True)
    rec_1 = report['1']['recall']
    prec_1 = report['1']['precision']
    
    print(f"Threshold: {threshold:.2f} | Recall (class 1): {rec_1:.2f} | Precision (class 1): {prec_1:.2f}")

Threshold: 0.30 | Recall (class 1): 0.83 | Precision (class 1): 0.72
Threshold: 0.35 | Recall (class 1): 0.83 | Precision (class 1): 0.75
Threshold: 0.40 | Recall (class 1): 0.82 | Precision (class 1): 0.77
Threshold: 0.45 | Recall (class 1): 0.82 | Precision (class 1): 0.79
Threshold: 0.50 | Recall (class 1): 0.82 | Precision (class 1): 0.81
Threshold: 0.55 | Recall (class 1): 0.81 | Precision (class 1): 0.82


In [22]:
for threshold in [0.1, 0.15, 0.2, 0.25]:
    y_pred_custom = (probs_balanced > threshold).astype(int)
    
    report = classification_report(y_test, y_pred_custom, output_dict=True)
    rec_1 = report['1']['recall']
    prec_1 = report['1']['precision']
    
    print(f"Threshold: {threshold:.2f} | Recall (class 1): {rec_1:.2f} | Precision (class 1): {prec_1:.2f}")

Threshold: 0.10 | Recall (class 1): 0.89 | Precision (class 1): 0.48
Threshold: 0.15 | Recall (class 1): 0.88 | Precision (class 1): 0.61
Threshold: 0.20 | Recall (class 1): 0.87 | Precision (class 1): 0.67
Threshold: 0.25 | Recall (class 1): 0.84 | Precision (class 1): 0.71


In [23]:
best_threshold = 0.15
y_pred_final = (probs_balanced > best_threshold).astype(int)

print(f"--- Final metrics (Threshold {best_threshold}) ---")
print(classification_report(y_test, y_pred_final))

--- Final metrics (Threshold 0.15) ---
              precision    recall  f1-score   support

           0       0.98      0.91      0.94       859
           1       0.61      0.88      0.72       141

    accuracy                           0.90      1000
   macro avg       0.79      0.89      0.83      1000
weighted avg       0.93      0.90      0.91      1000

